# 03 - Separacao entre teste revisado e treino automatico

A anotacao de referencia deste trabalho tem duas origens, e a separacao entre
elas e o que torna a avaliacao interpretavel:

- **conjunto de teste**: revisado manualmente, titulo a titulo. E a unica
  referencia confiavel, e todos os resultados sao medidos contra ele.
- **conjunto de treino**: rotulado pelas regras do notebook 02, sem revisao.
  E barato e imperfeito.

Essa divisao permite medir uma coisa que a divisao usual nao mede: **quanto um
modelo perde por ter sido treinado com rotulo automatico**. Se o modelo treinado
em rotulo automatico se aproximar do desempenho medido contra o teste revisado,
a supervisao fraca se justifica; se ficar muito abaixo, o custo da anotacao
manual esta demonstrado com numero.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import collections
import random

import pandas as pd

from src.dados import (
    carregar_produtos, titulos_unicos, marcas_do_catalogo,
    salvar_jsonl, carregar_jsonl, DIR_ANOT,
)

BASE = "cellphone"

from src.preanotacao import anotar_base

TAMANHO_TESTE = 50
SEMENTE = 42

In [ ]:
produtos = titulos_unicos(carregar_produtos(BASE))
preanotado = anotar_base(produtos, marcas_do_catalogo(produtos))
print(f"titulos disponiveis: {len(preanotado)}")

## 1. Amostragem estratificada do teste

O teste precisa refletir a composicao da base, senao o resultado fica enviesado
para a categoria mais frequente. A amostra preserva a proporcao entre
smartphones e acessorios.

In [ ]:
random.seed(SEMENTE)

por_categoria = collections.defaultdict(list)
for indice, produto in enumerate(produtos):
    por_categoria[produto["categoria"] or "sem categoria"].append(indice)

indices_teste = []
for categoria, indices in por_categoria.items():
    cota = round(len(indices) * TAMANHO_TESTE / len(produtos))
    if cota:
        indices_teste.extend(random.sample(indices, min(cota, len(indices))))

indices_teste = sorted(indices_teste)
indices_treino = [i for i in range(len(produtos)) if i not in set(indices_teste)]

resumo = pd.DataFrame([
    {"categoria": categoria,
     "base": len(indices),
     "teste": sum(1 for i in indices_teste if i in set(indices))}
    for categoria, indices in sorted(por_categoria.items(), key=lambda x: -len(x[1]))
])
print(f"teste: {len(indices_teste)} | treino: {len(indices_treino)}")
resumo

## 2. Arquivos gerados

`teste_revisar.jsonl` vai para a ferramenta de anotacao em `tools/anotador.html`.
Depois da revisao, o arquivo exportado precisa ser salvo como
`cellphone.ibyte.teste.jsonl` para que os notebooks seguintes o encontrem.

In [ ]:
salvar_jsonl([preanotado[i] for i in indices_teste], DIR_ANOT / f"{BASE}.ibyte.teste_revisar.jsonl")
salvar_jsonl([preanotado[i] for i in indices_treino], DIR_ANOT / f"{BASE}.ibyte.treino_auto.jsonl")

print("gerados:")
print(f"  {BASE}.ibyte.teste_revisar.jsonl  ({len(indices_teste)} titulos, para revisao manual)")
print(f"  {BASE}.ibyte.treino_auto.jsonl    ({len(indices_treino)} titulos, rotulo automatico)")

## 3. Verificacao apos a revisao

Rode a celula abaixo depois de salvar o arquivo revisado. Ela confere se o
formato esta valido e mostra o quanto a revisao alterou a anotacao automatica.
Essa diferenca e um resultado do trabalho: mede a taxa de erro das regras.

In [ ]:
caminho_teste = DIR_ANOT / f"{BASE}.ibyte.teste.jsonl"

if not caminho_teste.exists():
    print("arquivo revisado ainda nao existe; rode esta celula depois da revisao")
else:
    teste = carregar_jsonl(caminho_teste)
    automatico = {r["text"]: r["entities"] for r in carregar_jsonl(DIR_ANOT / f"{BASE}.ibyte.teste_revisar.jsonl")}

    iguais = sum(1 for r in teste
                 if sorted(map(tuple, r["entities"])) == sorted(map(tuple, automatico.get(r["text"], []))))

    print(f"titulos no teste revisado: {len(teste)}")
    print(f"titulos que a revisao nao alterou: {iguais} ({iguais / len(teste):.1%})")
    print(f"entidades apos revisao: {sum(len(r['entities']) for r in teste)}")
    print(f"entidades antes da revisao: {sum(len(e) for e in automatico.values())}")

    from src.avaliacao import avaliar
    referencia = teste
    predicao = [{"text": r["text"], "entities": automatico.get(r["text"], [])} for r in teste]
    pd.DataFrame(avaliar(referencia, predicao))